# Laboratorio 4 — Enunciado: Temperaturas CRU

## Antes de empezar

El proyecto ya está preparado. Trabajen en parejas y ejecuten los comandos
desde la carpeta de este laboratorio:

```bash
uv sync
uv run jupyter lab
```

El notebook es el espacio para explorar los datos y probar las
transformaciones. El código definitivo debe quedar en `src/meteolab/`.

## Flujo de trabajo

En cada operación seguirán este ciclo:

1. Investigar la operación o el método que necesitan.
2. Probarlo directamente sobre los datos en el notebook.
3. Observar y comprobar el resultado.
4. Trasladar la solución al módulo indicado.
5. Comprobarla con `uv run pytest -m etapaN`.
6. Interpretar los resultados y responder la pregunta de la etapa.

## El dataset CRU

El archivo `data/cru_country_tmp_tidy.csv` contiene temperaturas medias del
conjunto de datos de la *Climatic Research Unit* (CRU). Cada fila representa
un país, un año y un período. El período puede ser un mes (`JAN` a `DEC`),
una estación climática (`DJF`, `MAM`, `JJA`, `SON`) o el promedio anual
(`ANN`).

| Columna | Significado | Tipo esperado |
|---|---|---|
| `country` | nombre del país | `String` |
| `iso_alpha2` | código ISO de dos letras | `String` |
| `iso_alpha3` | código ISO de tres letras | `String` |
| `year` | año de la observación | `Int64` |
| `period` | mes, estación climática o promedio anual | `String` |
| `temperature_c` | temperatura media en grados Celsius | `Float64` |
| `parameter` | indicador medido | `String` |
| `units` | unidad del indicador | `String` |
| `source_file` | archivo de origen | `String` |

El laboratorio se concentrará en las **temperaturas medias mensuales**.
Durante la limpieza descartarán las filas de `DJF`, `MAM`, `JJA`, `SON` y
`ANN`. Desde ese punto, ninguna agregación podrá usar esas observaciones.

```mermaid
flowchart LR
    A["CSV CRU<br/>17 períodos"] --> B["Exploración"]
    B --> C["Lectura y esquema"]
    C --> D["Limpieza<br/>solo JAN–DEC"]
    D --> E["Fechas mensuales"]
    E --> F["Agregaciones y ventanas"]
    F --> G["Pipeline lazy"]
```

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — Polars</strong>
  <strong>Polars</strong> es una librería de Python para trabajar con datos tabulares. Sus estructuras principales son <code>DataFrame</code>, que contiene datos ya materializados, y <code>LazyFrame</code>, que representa una consulta aún no ejecutada. Polars ofrece una API de expresiones para describir transformaciones sobre columnas y permite trabajar en modo <em>eager</em> o <em>lazy</em>.
</div>

## Evaluación

| Etapa | Contenido | Implementación | Análisis y preguntas | Total |
|---|---|---:|---:|---:|
| 1 | Exploración del CSV | 0.2 | 0.4 | 0.6 |
| 2 | Polars e I/O | 0.4 | 0.3 | 0.7 |
| 3 | Esquema y validación | 0.5 | 0.3 | 0.8 |
| 4 | Limpieza y selección mensual | 0.5 | 0.4 | 0.9 |
| 5 | Fechas y agregaciones | 0.7 | 0.4 | 1.1 |
| 6 | Ventanas y anomalías | 0.5 | 0.3 | 0.8 |
| 7 | Pipeline lazy y análisis | 0.3 | 0.8 | 1.1 |
| **Total** | | **3.1** | **2.9** | **6.0** |

## Preparación

In [ ]:
import sys
from pathlib import Path

import plotly.express as px
import polars as pl

RAIZ = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

RUTA_DATOS = RAIZ / "data"
RUTA_CSV = RUTA_DATOS / "cru_country_tmp_tidy.csv"

print("Proyecto:", RAIZ)
print("Archivo :", RUTA_CSV)

---
# Etapa 1 — Explorar el archivo (0.6 puntos)

Antes de decidir cómo leer o transformar una tabla, observen sus dimensiones,
sus tipos y sus valores ausentes. Esta etapa no modifica los datos: levanta
evidencia para las decisiones que tomarán después.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — tabla y esquema</strong>
  Una tabla organiza observaciones en filas y variables en columnas. El <strong>esquema</strong> es la lista de columnas junto con sus tipos. En este archivo, el esquema distingue identificadores de texto, años enteros y temperaturas decimales.
</div>

### 1.1 — Leer e inspeccionar (0.1 puntos)

Lean el CSV sin imponer todavía el esquema. En la siguiente etapa justificarán
qué tipos y valores nulos deben declarar explícitamente.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — CSV</strong>
  <strong>CSV</strong> (*Comma-Separated Values*) es un archivo de texto en el que cada fila representa un registro y las columnas se separan mediante un delimitador, normalmente una coma. Es fácil de intercambiar, pero no guarda de forma completa el esquema de la tabla: al leerlo, la librería debe inferir o recibir los tipos de cada columna.
</div>

Investiguen `polars.read_csv` y utilícenla para leer `RUTA_CSV`.

In [ ]:
raw = pl.read_csv(RUTA_CSV)

Inspeccionen las primeras y las últimas diez filas. Luego, muestren diez filas
aleatorias con una semilla fija.

In [ ]:
display(raw.head(10))
display(raw.tail(10))
display(raw.sample(n=10, seed=7202))

Revisen las dimensiones, los nombres de las columnas, el esquema y una vista
compacta de los valores.

In [ ]:
print("Dimensiones:", raw.shape)
print("Columnas:", raw.columns)
print("Esquema:")
print(raw.schema)
raw.glimpse()

Generen un resumen estadístico y cuenten los valores nulos por columna.

In [ ]:
display(raw.describe())
display(raw.null_count())

### 1.2 — Visualizar la distribución (0.1 puntos)

Plotly Express permite construir gráficos interactivos a partir de columnas
tabulares. Exploren la distribución de `temperature_c` sin confundir los
valores ausentes con una temperatura.

In [ ]:
fig = px.histogram(
    raw,
    x="temperature_c",
    nbins=40,
    title="Distribución de las temperaturas CRU",
    labels={"temperature_c": "Temperatura (°C)"},
)
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 1 — Qué muestra el archivo (0.2 puntos)</strong>
  ¿Cuántas filas y columnas tiene el CSV? ¿Qué columnas son identificadores, cuáles representan tiempo y cuál contiene la medición? Expliquen qué información se pierde si `temperature_c` se lee como texto.
</div>

El CSV tiene 408.000 filas y 9 columnas. Las columnas `country`, `iso_alpha2` e `iso_alpha3` identifican el país y `parameter`, `units` y `source_file` entregan metadatos sobre la medición y su origen. Las variables temporales son `year` y `period`, mientras que `temperature_c` contiene la medición de temperatura media en grados Celsius.

Si `temperature_c` se leyera como texto, se perdería su información numérica, no podríamos calcular correctamente medias, desviaciones estándar, histogramas o comparaciones aritméticas sin convertir antes la columna. Además, el ordenamiento podría verse en orden alfabético en vez de numérico.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 2 — Escalas temporales (0.2 puntos)</strong>
  El archivo contiene 17 valores distintos en `period`. ¿Por qué no se deben mezclar en un mismo promedio las filas mensuales, estacionales y anuales? Anticipen qué período conservarán durante la limpieza y por qué.
</div>

No se deben mezclar los 17 períodos porque representan escalas temporales distintas: 12 meses, 4 promedios estacionales y 1 promedio anual. Incluirlos juntos en una misma media contaría varias veces información del mismo año y daría el mismo peso a observaciones con distinta duración temporal.

Durante la limpieza se conservará únicamente `JAN` a `DEC`, porque el objetivo del laboratorio es analizar temperaturas medias mensuales y construir desde ellas las agregaciones.

---
# Etapa 2 — Polars e I/O (0.7 puntos)

En esta etapa pasarán de una lectura exploratoria a una lectura reproducible.
Usarán el CSV como fuente de entrada y compararán la lectura eager con la
lectura lazy.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — eager y lazy</strong>
  En modo <em>eager</em>, una operación se ejecuta cuando se llama y devuelve un <code>DataFrame</code>. En modo <em>lazy</em>, las operaciones construyen un plan y se ejecutan al llamar <code>collect()</code>. <code>scan_csv</code> permite describir una consulta sin cargar de inmediato todo el archivo.
</div>

### 2.1 — Investigar y probar la lectura CSV (0.3 puntos)

Investiguen `schema_overrides` en `read_csv`. Luego, vuelvan a leer el CSV
declarando `year` como `Int64` y `temperature_c` como `Float64`. Comparen el
esquema de esta tabla con el de `raw`.

In [ ]:
lecturas = pl.read_csv(
    RUTA_CSV,
    schema_overrides={
        "year": pl.Int64,
        "temperature_c": pl.Float64,
    },
)

In [ ]:
print("Esquema inferido :", raw.schema)
print("Esquema declarado:", lecturas.schema)

Investiguen `scan_csv` y construyan una consulta que filtre algunos países
sin ejecutarla. Comprueben el tipo de objeto y materialicen el resultado con
`collect()`.

In [ ]:
consulta = pl.scan_csv(
    RUTA_CSV,
    schema_overrides={
        "year": pl.Int64,
        "temperature_c": pl.Float64,
    },
).filter(pl.col("iso_alpha3").is_in(["CHL", "ARG", "PER"]))

In [ ]:
print(type(consulta))
display(consulta.collect().head())
print(consulta.explain())

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 3 — Elegir la lectura (0.3 puntos)</strong>
  ¿Qué ventaja ofrece declarar `year` y `temperature_c` al leer el CSV? ¿En qué situación tendría sentido usar `scan_csv` en vez de `read_csv`?
</div>

Declarar `year` como `Int64` y `temperature_c` como `Float64` hace explícito el tipo de variable y evita que una inferencia dependiente de los valores observados cambie el comportamiento del análisis. También permite detectar antes datos incompatibles con esos tipos.

`scan_csv` tiene sentido cuando el archivo es grande o cuando solo necesitamos una parte de sus filas o columnas. Al trabajar en modo lazy, Polars puede optimizar el plan y aplicar filtros/proyecciones antes de entregar el resultado. `read_csv` es más cómodo cuando el archivo es pequeño y queremos ver inmediatamente el `DataFrame` completo.

### 2.2 — Trasladar la lectura CSV al módulo (0.1 puntos)

Después de probar las operaciones, implementen en `src/meteolab/carga.py`:

- `leer_temperaturas(ruta)`, con `schema_overrides`;
- `escanear_temperaturas(ruta)`, que debe devolver un `LazyFrame`;
Comprueben también que `escanear_temperaturas` devuelve un `LazyFrame` y que
la consulta se materializa solo con `collect()`.

In [ ]:
from src.meteolab.carga import (
    escanear_temperaturas,
    leer_temperaturas,
)

lecturas_modulo = leer_temperaturas(RUTA_CSV)
consulta_modulo = escanear_temperaturas(RUTA_CSV)

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240, 136, 62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Al módulo — <code>src/meteolab/carga.py</code></strong>
  Trasladen las funciones de lectura al módulo. Comprueben con <code>uv run pytest -m etapa2</code> que los tipos, los nulos y el modo lazy se comporten como en el notebook.
</div>

---
# Etapa 3 — Tipos y validación (0.8 puntos)

Leer un archivo no basta: hay que comprobar que los nombres, tipos y valores
permitidos cumplen el contrato del dataset.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — inferir, forzar y validar</strong>
  <strong>Inferir</strong> es dejar que la librería decida un tipo a partir de los valores observados. <strong>Forzar</strong> es declarar el tipo esperado al leer o transformar. <strong>Validar</strong> es comprobar que, además del tipo, los valores cumplen restricciones del dominio.
</div>

### 3.1 — Investigar y declarar el esquema (0.3 puntos)

Investiguen los tipos de Polars y construyan en el notebook un diccionario
con el esquema esperado. Comparen ese diccionario con `lecturas.schema` y
expliquen cualquier diferencia.

In [ ]:
esquema_notebook = {
    "country": pl.String,
    "iso_alpha2": pl.String,
    "iso_alpha3": pl.String,
    "year": pl.Int64,
    "period": pl.String,
    "temperature_c": pl.Float64,
    "parameter": pl.String,
    "units": pl.String,
    "source_file": pl.String,
}

In [ ]:
diferencias = {
    columna: (lecturas.schema.get(columna), tipo)
    for columna, tipo in esquema_notebook.items()
    if lecturas.schema.get(columna) != tipo
}
print("Diferencias:", diferencias)

Investiguen Pandera y definan las restricciones que corresponden a este
dataset: años entre 1901 y 2025, períodos permitidos, `Mean Temperature` y
`degrees Celsius`.

In [ ]:
from src.meteolab.constantes import PERIODOS_VALIDOS

In [ ]:
import pandera.polars as pa

validacion_notebook = pa.DataFrameSchema(
    {
        "country": pa.Column(pl.String),
        "iso_alpha2": pa.Column(pl.String),
        "iso_alpha3": pa.Column(pl.String),
        "year": pa.Column(
            pl.Int64,
            checks=[pa.Check.ge(1901), pa.Check.le(2025)],
        ),
        "period": pa.Column(
            pl.String,
            checks=pa.Check.isin(PERIODOS_VALIDOS),
        ),
        "temperature_c": pa.Column(pl.Float64, nullable=True),
        "parameter": pa.Column(
            pl.String,
            checks=pa.Check.equal_to("Mean Temperature"),
        ),
        "units": pa.Column(
            pl.String,
            checks=pa.Check.equal_to("degrees Celsius"),
        ),
        "source_file": pa.Column(pl.String),
    },
    strict=True,
)

validacion_notebook.validate(lecturas, lazy=True)

Los 17 valores de `period` pertenecen al contrato del CSV. Que existan en el
esquema no significa que todos vayan a entrar al análisis: la selección de
períodos mensuales se hará en la limpieza.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 4 — El contrato y el análisis (0.2 puntos)</strong>
  ¿Por qué conviene aceptar `DJF`, `MAM`, `JJA` y `ANN` al validar el archivo, pero excluirlos después durante la limpieza? Relacionen la respuesta con la diferencia entre validar una fuente y definir el universo del análisis.
</div>

Conviene aceptar `DJF`, `MAM`, `JJA`, `SON` y `ANN` durante la validación porque esos valores sí pertenecen al contrato de la fuente CRU. Rechazarlos en esa etapa haría parecer inválido un archivo que está correctamente estructurado.

La limpieza es otra pregunta: define el universo que se usará en el análisis. Como queremos trabajar con temperaturas mensuales, después de validar la fuente seleccionamos solo `JAN` a `DEC`. Validar comprueba que la fuente sea válida y limpiar decide qué parte válida de esa fuente sirve para el análisis.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 5 — Validar tipos y valores (0.1 puntos)</strong>
  ¿Qué aporta Pandera, además de comparar `lecturas.schema` con el esquema esperado? Mencionen una restricción de valores que Pandera pueda comprobar en este dataset.
</div>

Pandera permite validar no solo nombres y tipos, sino también restricciones de dominio sobre los valores y reportar casos que incumplen el contrato. Por ejemplo, puede comprobar que `year` esté entre 1901 y 2025, que `period` pertenezca a los 17 períodos permitidos o que `units` sea siempre `degrees Celsius`.

### 3.2 — Trasladar la validación al módulo (0.2 puntos)

Después de probar el esquema y sus restricciones, implementen en
`src/meteolab/esquema.py`:

- `comparar_esquema`, para informar columnas faltantes y tipos distintos;
- `validar_esquema`, para rechazar un esquema incorrecto;
- `ESQUEMA_TEMPERATURAS`, con Pandera;
- `validar_datos` y `casos_que_fallan`.

In [ ]:
from src.meteolab.esquema import (
    ESQUEMA_TEMPERATURAS,
    casos_que_fallan,
    validar_datos,
    validar_esquema,
)

validar_esquema(lecturas_modulo)
validado = validar_datos(lecturas_modulo)
print("Filas validadas:", validado.height)
print(ESQUEMA_TEMPERATURAS)

In [ ]:
muestra_invalida = lecturas_modulo.head(3).with_columns(
    pl.lit("XYZ").alias("period")
)
fallas = casos_que_fallan(muestra_invalida)
display(fallas)

---
# Etapa 4 — Limpieza: conservar solo meses (0.9 puntos)

Esta es la decisión central del laboratorio. El archivo mezcla tres escalas
temporales. Desde este punto trabajarán solo con las temperaturas medias de
`JAN` a `DEC`.

In [ ]:
from src.meteolab.constantes import (
    PERIODOS_MENSUALES,
)

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Contrato de limpieza</strong>
  La tabla limpia debe contener únicamente los 12 períodos mensuales. Debe descartar <code>PERIODOS_ESTACIONALES = ("DJF", "MAM", "JJA", "SON")</code> y <code>PERIODO_ANUAL = "ANN"</code>. En este dataset, el nulo conocido pertenece a <code>DJF</code> de 2025 y desaparece al aplicar la selección mensual.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — filtrar filas</strong>
  Filtrar una tabla significa conservar las filas que cumplen una condición. Una condición sobre <code>period</code> define el universo temporal del análisis; no es lo mismo que corregir un valor ni que eliminar una columna.
</div>

### 4.1 — Investigar y limpiar (0.3 puntos)

Investiguen cómo combinar condiciones con `&` y cómo comprobar nulos. Luego,
filtren en el notebook la tabla para conservar solo `PERIODOS_MENSUALES` y
valores disponibles de `temperature_c`. Comprueben las filas y el esquema
resultantes.

In [ ]:
limpias = lecturas.filter(
    pl.col("period").is_in(PERIODOS_MENSUALES)
    & pl.col("temperature_c").is_not_null()
)

In [ ]:
display(limpias.null_count())

In [ ]:
assert set(limpias["period"].unique()) <= {
    "JAN",
    "FEB",
    "MAR",
    "APR",
    "MAY",
    "JUN",
    "JUL",
    "AUG",
    "SEP",
    "OCT",
    "NOV",
    "DEC",
}
assert limpias["temperature_c"].null_count() == 0
print("Filas mensuales limpias:", limpias.height)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 6 — No mezclar escalas (0.1 puntos)</strong>
  ¿Qué problema produciría calcular una media por país usando a la vez meses, estaciones y `ANN`? Expliquen qué filas quedan en la tabla limpia y qué representa cada una.
</div>

Si se promediaran simultáneamente meses, estaciones y `ANN`, cada año aportaría observaciones que representan horizontes temporales diferentes y parte de la información mensual quedaría contada nuevamente dentro de los promedios estacionales y anuales. 

Después de la limpieza quedan 288.000 filas: una observación por país, año y mes (`JAN`–`DEC`), con `temperature_c` disponible. Cada fila representa la temperatura media mensual de un país en un mes y año determinados.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 7 — Comprobar la completitud mensual (0.1 puntos)</strong>
  Después de la limpieza, ¿cómo comprobarían que cada combinación de país y año tiene doce observaciones mensuales? ¿Qué decisión tomarían si faltara un mes?
</div>

Agruparíamos por `iso_alpha3` y `year` y contaríamos las filas, verificando que el conteo sea 12 para todas las combinaciones. También se podría verificar que los 12 valores de `period` sean distintos.

Si faltara un mes, marcaríamos ese país-año como incompleto y revisaríamos la causa. Para comparaciones anuales que requieren años completos, lo excluiríamos mediante `meses_disponibles == 12` en vez de imputar automáticamente un valor sin una justificación.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 8 — Imputar con el promedio (0.2 puntos)</strong>
  El nulo conocido pertenece a `DJF` de 2025. Calculen el promedio global de `temperature_c` y supongan que, antes de filtrar los períodos, reemplazan ese valor por dicho promedio. ¿Qué valor se asignaría, qué pasaría con la tabla y qué problema introduciría esa imputación? Expliquen por qué en este laboratorio es preferible descartar la fila.
</div>

El promedio global de `temperature_c` en el archivo crudo es aproximadamente 19,06 °C. Si imputáramos el nulo de `DJF` 2025 con ese promedio, la tabla seguiría teniendo 408.000 filas y `temperature_c` dejaría de contener ese nulo.

Sin embargo, la imputación introduce un valor artificial que mezcla países, años y escalas temporales muy distintas, por lo que no representa necesariamente el `DJF` faltante de ese país. Además, como `DJF` no forma parte del análisis mensual, esa fila será descartada durante la limpieza. Por eso es preferible no inventar una medición y simplemente excluirla junto a los demás períodos.

### 4.2 — Trasladar la limpieza al módulo (0.2 puntos)

Después de probar las expresiones en el notebook, implementen en
`src/meteolab/limpieza.py`:

- `limpiar_temperaturas`, compatible con `DataFrame` y `LazyFrame`;
- `resumen_de_nulos`, para revisar los valores faltantes;
- `claves_repetidas`, para detectar repeticiones de país, año y período.

Las tres funciones se utilizan en las pruebas de esta etapa.

In [ ]:
from src.meteolab.limpieza import limpiar_temperaturas, resumen_de_nulos

limpias_modulo = limpiar_temperaturas(lecturas_modulo)
display(resumen_de_nulos(lecturas_modulo))

---
# Etapa 5 — Fechas y agregaciones mensuales (1.1 puntos)

`year` y `period` son columnas separadas. Construirán una fecha para ordenar
las observaciones y luego resumirán los años disponibles por mes.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — columna derivada</strong>
  Una columna derivada se calcula a partir de columnas existentes. La columna <code>fecha</code> no agrega una medición: combina <code>year</code> y el número del mes para permitir ordenamiento y gráficos temporales.
</div>

### 5.1 — Investigar y construir fechas (0.2 puntos)

Investiguen cómo traducir los códigos `JAN` a `DEC` a números de mes y cómo
convertir una cadena con formato ISO en una fecha de Polars. Construyan en el
notebook las columnas `month` y `fecha`, y comprueben que el resultado se
ordena cronológicamente.

In [ ]:
from src.meteolab.constantes import MESES

In [ ]:
mensuales = (
    limpias.with_columns(
        pl.col("period")
        .replace_strict(MESES, return_dtype=pl.Int8)
        .alias("month")
    )
    .with_columns(
        pl.date(pl.col("year"), pl.col("month"), 1).alias("fecha")
    )
    .sort("iso_alpha3", "fecha")
)

display(mensuales.head())

### 5.2 — Investigar y calcular resúmenes (0.4 puntos)

El resultado de esta exploración se trasladará después a
`src/meteolab/metricas.py`, en `resumen_mensual`. Debe devolver una fila por
país y mes, con:

- `iso_alpha3` y `country`;
- `month`;
- `observaciones`;
- `temperature_mean`, redondeada a dos decimales.

Antes de escribir la función, investiguen `group_by` y `agg`. Calculen el
resumen directamente en el notebook y revisen algunas filas.

In [ ]:
climatologia = (
    mensuales.group_by("iso_alpha3", "country", "month")
    .agg(
        pl.len().alias("observaciones"),
        pl.col("temperature_c")
        .mean()
        .round(2)
        .alias("temperature_mean"),
    )
    .sort("iso_alpha3", "month")
)

display(climatologia.head())

In [ ]:
PAISES = ["CHL", "ARG", "PER", "BOL", "BRA", "CAN", "EGY"]
fig = px.line(
    climatologia.filter(pl.col("iso_alpha3").is_in(PAISES)),
    x="month",
    y="temperature_mean",
    color="country",
    markers=True,
    title="Climatología mensual de países seleccionados",
    labels={
        "month": "Mes",
        "temperature_mean": "Temperatura media (°C)",
        "country": "País",
    },
)
fig.show()

Calculen también una media por año, pero debe salir de las filas mensuales
limpias. No usen las filas `ANN` del CSV. El resumen debe conservar los años
incompletos y la columna `meses_disponibles`; para comparar períodos, filtren
después los años cuyo conteo sea igual a 12.

In [ ]:
anuales_desde_meses = (
    mensuales.group_by("iso_alpha3", "country", "year")
    .agg(
        pl.len().alias("meses_disponibles"),
        pl.col("temperature_c")
        .mean()
        .round(2)
        .alias("temperature_mean"),
    )
    .sort("iso_alpha3", "year")
)

display(anuales_desde_meses.head())

In [ ]:
anuales_completos = anuales_desde_meses.filter(
    pl.col("meses_disponibles") == 12
)
print("Años completos:", anuales_completos.height)

Revisen si la tabla contiene más de una observación para la misma combinación
de país, año y período. Investiguen `group_by` y `len` para contar posibles
repeticiones de esa clave.

In [ ]:
repetidas = (
    mensuales.group_by("iso_alpha3", "year", "period")
    .len()
    .filter(pl.col("len") > 1)
    .sort("iso_alpha3", "year", "period")
)

display(repetidas)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 9 — Qué significa una climatología mensual (0.2 puntos)</strong>
  En `climatologia`, ¿qué representa una fila para `CHL` y `month = 1`? ¿Por qué esa fila resume varios años y no un solo registro del archivo?
</div>

Una fila con `iso_alpha3 = "CHL"` y `month = 1` representa la climatología histórica de enero para Chile. En este dataset resume 125 observaciones de enero, una por cada año entre 1901 y 2025, y su `temperature_mean` es de 13,08 °C.

Por eso no corresponde a una sola fila del CSV: `group_by` reúne todos los eneros de Chile y calcula una estadística sobre ese conjunto histórico.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 10 — Detectar duplicados (0.2 puntos)</strong>
  ¿Qué combinación de columnas debería identificar de forma única una observación mensual? ¿Cómo detectarían registros duplicados antes de calcular las métricas?
</div>

La combinación `iso_alpha3`, `year` y `period` debería identificar de forma única una observación mensual. Antes de calcular métricas, agruparíamos por esas tres columnas, aplicaríamos `len()` y filtraríamos los grupos con `len > 1`. Cualquier fila resultante indicaría una clave duplicada que debe investigarse antes de agregar los datos.

---

### 5.3 — Trasladar las transformaciones al módulo (0.1 puntos)

Después de probar las expresiones, implementen `agregar_fecha_mensual`,
`resumen_mensual` y `resumen_anual_desde_mensuales` en sus módulos. Comparen
los resultados del módulo con los que obtuvieron directamente en el
notebook.

In [ ]:
from src.meteolab.derivadas import agregar_fecha_mensual
from src.meteolab.metricas import (
    resumen_anual_desde_mensuales,
    resumen_mensual,
)
from src.meteolab.limpieza import claves_repetidas

In [ ]:
mensuales_modulo = agregar_fecha_mensual(limpias_modulo)
climatologia_modulo = resumen_mensual(mensuales_modulo)
anuales_modulo = resumen_anual_desde_mensuales(mensuales_modulo, ["CHL"])
repetidas_modulo = claves_repetidas(mensuales_modulo)

---
# Etapa 6 — Ventanas y anomalías mensuales (0.8 puntos)

Una media histórica de enero no debe compararse con una de julio. Usarán una
ventana por país y mes para medir cuánto se aparta cada observación de sus
pares comparables.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — ventana</strong>
  Una expresión con <code>.over("iso_alpha3", "month")</code> calcula una medida dentro de cada grupo y devuelve ese resultado en las filas originales. A diferencia de <code>group_by().agg()</code>, una ventana conserva una fila por observación.
</div>

### 6.1 — Investigar y calcular una ventana (0.4 puntos)

Investiguen `over` y construyan directamente en el notebook las tres
columnas solicitadas. Usen la dupla `(iso_alpha3, month)` como grupo de
comparación para que cada mes se compare con otros del mismo país.

Implementen `anomalias_mensuales(mensuales, umbral=2.0)` en el módulo después
de comprobar el resultado de esta exploración. Debe agregar:

- `temperature_mean_month`, la media histórica del país para ese mes;
- `standardized_anomaly`, la diferencia dividida por la desviación estándar;
- `is_anomaly`, booleana y sin nulos.

In [ ]:
grupos = ["iso_alpha3", "month"]
marcadas = (
    mensuales.with_columns(
        pl.col("temperature_c")
        .mean()
        .over(grupos)
        .alias("temperature_mean_month"),
        pl.col("temperature_c")
        .std()
        .over(grupos)
        .alias("temperature_std_month"),
    )
    .with_columns(
        pl.when(
            pl.col("temperature_std_month").is_not_null()
            & (pl.col("temperature_std_month") != 0)
        )
        .then(
            (pl.col("temperature_c") - pl.col("temperature_mean_month"))
            / pl.col("temperature_std_month")
        )
        .otherwise(0.0)
        .alias("standardized_anomaly")
    )
    .with_columns(
        (pl.col("standardized_anomaly").abs() > 2.0)
        .fill_null(False)
        .alias("is_anomaly")
    )
    .drop("temperature_std_month")
)

In [ ]:
fig = px.scatter(
    marcadas.filter(pl.col("iso_alpha3").is_in(PAISES)),
    x="fecha",
    y="standardized_anomaly",
    color="is_anomaly",
    facet_row="iso_alpha3",
    hover_data=["country", "period", "temperature_c"],
    title="Anomalías mensuales",
    labels={
        "fecha": "Fecha",
        "standardized_anomaly": "Anomalía estandarizada",
        "is_anomaly": "¿Anómala?",
    },
)
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 11 — Elegir el grupo de comparación (0.3 puntos)</strong>
  ¿Qué diferencia habría entre calcular la anomalía sobre `iso_alpha3` y calcularla sobre `(iso_alpha3, month)`? ¿Cuál de las dos opciones permite distinguir mejor una variación inusual de una diferencia normal entre estaciones del año?
</div>

Si la ventana se definiera solo por `iso_alpha3`, cada observación se compararía con todos los meses del país. En países con estacionalidad muy marcada, un mes de verano típicamente cálido podría parecer fuera de la norma frente a la media anual, y un mes típicamente frío podría producir el efecto contrario.

Al usar `(iso_alpha3, month)`, cada enero se compara con otros eneros del mismo país, cada febrero con otros febreros, y así sucesivamente. Esta segunda opción controla la estacionalidad y permite distinguir mucho mejor una variación inusual de una diferencia normal entre estaciones.

---

### 6.2 — Trasladar la ventana al módulo (0.1 puntos)

Después de revisar el resultado, implementen `anomalias_mensuales` en
`src/meteolab/metricas.py`. Comparen las columnas y la cantidad de filas con
el cálculo realizado directamente en el notebook.

In [ ]:
from src.meteolab.metricas import anomalias_mensuales

marcadas_modulo = anomalias_mensuales(mensuales_modulo, umbral=2.0)

---
# Etapa 7 — Pipeline lazy y análisis (1.1 puntos)

Integrarán las funciones en una consulta lazy. El pipeline debe leer el CSV,
descartar los períodos no mensuales, construir las fechas y producir el
resumen pedido sin materializar pasos intermedios.

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Contrato del pipeline</strong>
  <code>pipeline_mensual</code> debe devolver un <code>LazyFrame</code>. El filtrado de países, la selección de meses y la construcción de columnas deben formar parte del plan. La ejecución ocurre solo con <code>collect()</code>.
</div>

### 7.1 — Investigar y construir una consulta lazy (0.1 puntos)

Investiguen cómo combinar `scan_csv`, `filter`, `with_columns` y `collect`.
Construyan primero una consulta lazy directamente en el notebook que:

- lea el CSV con los tipos esperados;
- conserve solo las filas mensuales con temperatura disponible;
- seleccione los países de `PAISES`;
- agregue `month` y `fecha`;
- ordene por país y fecha.

Comprueben que la consulta no se ejecuta hasta llamar a `collect()` y revisen
las primeras filas del resultado.

In [ ]:
consulta_notebook = (
    pl.scan_csv(
        RUTA_CSV,
        schema_overrides={
            "year": pl.Int64,
            "temperature_c": pl.Float64,
        },
    )
    .filter(
        pl.col("period").is_in(PERIODOS_MENSUALES)
        & pl.col("temperature_c").is_not_null()
        & pl.col("iso_alpha3").is_in(PAISES)
    )
    .with_columns(
        pl.col("period")
        .replace_strict(MESES, return_dtype=pl.Int8)
        .alias("month")
    )
    .with_columns(
        pl.date(pl.col("year"), pl.col("month"), 1).alias("fecha")
    )
    .sort("iso_alpha3", "fecha")
)

print(type(consulta_notebook))
display(consulta_notebook.collect().head())

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 12 — Elegir entre lazy y eager (0.2 puntos)</strong>
  Comparen esta consulta lazy con una solución eager que lea el archivo mediante `read_csv` y ejecute cada transformación de inmediato. ¿Qué ventajas ofrece construir un `LazyFrame` y ejecutar al final con `collect()`? Mencionen dos ventajas concretas y una situación en que preferirían trabajar en modo eager.
</div>

Un `LazyFrame` permite describir primero todo el flujo y ejecutarlo una sola vez con `collect()`. Algunas ventajas concretas son que Polars puede optimizar el plan, por ejemplo empujando filtros y selección de columnas hacia la lectura del CSV y evita materializar innecesariamente cada resultado intermedio, lo que puede reducir tiempo y memoria en archivos grandes.

Preferiríamos modo eager para una exploración pequeña o interactiva en la que el dataset cabe cómodamente en memoria y queremos inspeccionar inmediatamente el resultado de cada transformación.

### 7.2 — Trasladar el pipeline al módulo (0.2 puntos)

Después de probar la consulta, implementen en `src/meteolab/reporte.py`:

- `pipeline_mensual`;
- `pipeline_resumen_mensual`;
- `pipeline_resumen_anual`, calculado desde meses;
- `pipeline_anomalias`;
- `ejecutar_reporte` y `plan_de_ejecucion`.

In [ ]:
from src.meteolab.reporte import (
    ejecutar_reporte,
    pipeline_anomalias,
    pipeline_mensual,
    pipeline_resumen_anual,
    pipeline_resumen_mensual,
    plan_de_ejecucion,
)

consulta = pipeline_mensual(RUTA_CSV, PAISES)
print(type(consulta))
print(plan_de_ejecucion(RUTA_CSV, PAISES))
print("Filas directas:", consulta_notebook.collect().height)
print("Filas del módulo:", consulta.collect().height)

In [ ]:
reporte_mensual = ejecutar_reporte(RUTA_CSV, PAISES)
print(reporte_mensual.shape)
display(reporte_mensual.head())

In [ ]:
reporte_anual = pipeline_resumen_anual(RUTA_CSV, PAISES).collect()
anomalias = pipeline_anomalias(RUTA_CSV, ["CHL"]).collect()
print("Años calculados desde meses:", reporte_anual.height)
print("Anomalías marcadas:", anomalias["is_anomaly"].sum())

### 7.3 — Analizar cambios de temperatura a largo plazo (0.2 puntos)

Construyan primero una serie con la media anual de cada país y grafiquen cómo
evoluciona desde 1901. Después comparen la media de 1901–1930 con la de
1991–2020. Estas comparaciones describen cambios de temperatura; por sí solas
no demuestran sus causas.

In [ ]:
evolucion_anual = pipeline_resumen_anual(
    RUTA_CSV, PAISES
).collect()

display(evolucion_anual.head())

In [ ]:
fig = px.line(
    evolucion_anual,
    x="year",
    y="temperature_mean",
    color="country",
    title="Evolución de la temperatura media anual",
    labels={
        "year": "Año",
        "temperature_mean": "Temperatura media (°C)",
        "country": "País",
    },
)
fig.show()

Comparen ahora los dos períodos de referencia para resumir el cambio.

In [ ]:
antiguo = (
    evolucion_anual.filter(pl.col("year").is_between(1901, 1930))
    .group_by("iso_alpha3", "country")
    .agg(
        pl.col("temperature_mean")
        .mean()
        .alias("media_1901_1930")
    )
)

reciente = (
    evolucion_anual.filter(pl.col("year").is_between(1991, 2020))
    .group_by("iso_alpha3", "country")
    .agg(
        pl.col("temperature_mean")
        .mean()
        .alias("media_1991_2020")
    )
)

comparacion = (
    antiguo.join(reciente, on=["iso_alpha3", "country"])
    .with_columns(
        (pl.col("media_1991_2020") - pl.col("media_1901_1930"))
        .round(2)
        .alias("cambio_c")
    )
    .sort("cambio_c", descending=True)
)

display(comparacion)

In [ ]:
fig = px.bar(
    comparacion,
    x="country",
    y="cambio_c",
    color="cambio_c",
    title="Cambio de temperatura media entre períodos de referencia",
    labels={
        "country": "País",
        "cambio_c": "Cambio de temperatura (°C)",
    },
)
fig.update_xaxes(categoryorder="total descending")
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 13 — Cambios de temperatura y sus límites (0.2 puntos)</strong>
  Observen la evolución anual y la tabla <code>comparacion</code>. ¿Qué países presentan el mayor aumento entre los dos períodos? ¿Todos muestran el mismo cambio? Usen valores concretos para describir la tendencia y expliquen por qué este análisis aporta evidencia descriptiva para estudiar el calentamiento global, pero no permite atribuir causas ni calcular por sí solo una temperatura global.
</div>

Entre los países seleccionados, Canadá presenta el mayor aumento entre 1901–1930 y 1991–2020, de aprox +1,28 °C. Le sigue Egipto, +0,92 °C, y Brasil, +0,83 °C. Los cambios no son iguales: Argentina aumenta cerca de +0,47 °C, Perú +0,36 °C, Chile +0,29 °C y Bolivia solo alrededor de +0,08 °C.

Esto entrega evidencia de que las temperaturas medias de estos países cambiaron entre ambos períodos, pero no identifica la causa de esos cambios: para atribución causal se necesitarían otros datos y un diseño de análisis más específico. Tampoco permite calcular directamente una temperatura global, porque aquí estamos comparando promedios nacionales y no una agregación espacial ponderada por área, cobertura u otro criterio más especializado.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 14 — Interpretar el resultado (0.2 puntos)</strong>
  Elijan un país y describan dos patrones que observen en su climatología mensual o en sus anomalías. Usen valores concretos de las tablas o gráficos. Indiquen también qué información no se puede concluir a partir de este archivo.
</div>

En Chile se observan dos patrones. Primero, existe una estacionalidad clara: la climatología mensual alcanza aproximadamente 13,08 °C en enero y cae hasta cerca de 4,91 °C en julio, una diferencia de alrededor de 8,17 °C entre ambos promedios históricos. Segundo, existen meses que se apartan con fuerza de la climatología de su mismo mes, por ejemplo, febrero de 2024 registra aproximadamente 14,8 °C y una anomalía estandarizada cercana a +3,16, mientras enero de 1971 registra cerca de 11,2 °C y una anomalía de aprox −3,25.

El archivo permite describir patrones temporales de temperatura media a nivel de país, pero no permite inferir por sí solo las causas de las anomalías, su impacto local dentro de Chile ni variables meteorológicas adicionales como las precipitaciones o eventos extremos.

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240, 136, 62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Antes de entregar</strong>
  Ejecuten las pruebas por etapa y luego la suite completa:

  <pre><code>uv run pytest -m etapa1
uv run pytest -m etapa2
uv run pytest -m etapa3
uv run pytest -m etapa4
uv run pytest -m etapa5
uv run pytest -m etapa6
uv run pytest -m etapa7
uv run pytest</code></pre>

  Reinicien el kernel y ejecuten el notebook completo. Revisen que los gráficos Plotly se rendericen y que las respuestas usen resultados observables.
</div>